# Silver Layer — Clean & Enrich Education Data
Parses dates, derives GPA proxy, pass rates, retention risk flags, grade bands, and faculty load bands.

In [ ]:
from pyspark.sql.functions import (
    col, to_date, when, round as spark_round, current_timestamp,
    avg, count
)

df_faculty     = spark.read.format('delta').table('bronze_faculty')
df_students    = spark.read.format('delta').table('bronze_students')
df_enrolments  = spark.read.format('delta').table('bronze_enrolments')
df_assessments = spark.read.format('delta').table('bronze_assessments')

In [ ]:
# Silver faculty — cast numerics, derive load band
silver_faculty = (
    df_faculty
    .withColumn('hire_date',         to_date('hire_date'))
    .withColumn('courses_assigned',  col('courses_assigned').cast('int'))
    .withColumn('years_at_institution', col('years_at_institution').cast('int'))
    .withColumn('load_band',
        when(col('courses_assigned') >= 6, 'Heavy')
        .when(col('courses_assigned') >= 4, 'Moderate')
        .when(col('courses_assigned') >= 2, 'Light')
        .otherwise('Minimal')
    )
    .withColumn('silver_timestamp', current_timestamp())
)
silver_faculty.write.mode('overwrite').format('delta').saveAsTable('silver_faculty')
print(f'Silver faculty: {spark.table("silver_faculty").count()} rows')

In [ ]:
# Silver students — parse dates, derive status flags
silver_students = (
    df_students
    .withColumn('enrolment_date', to_date('enrolment_date'))
    .withColumn('age_at_enrolment', col('age_at_enrolment').cast('int'))
    .withColumn('cohort_year', col('cohort_year').cast('int'))
    .withColumn('is_active',
        when(col('status') == 'Active', 1).otherwise(0)
    )
    .withColumn('is_withdrawn',
        when(col('status') == 'Withdrawn', 1).otherwise(0)
    )
    .withColumn('silver_timestamp', current_timestamp())
)
silver_students.write.mode('overwrite').format('delta').saveAsTable('silver_students')
print(f'Silver students: {spark.table("silver_students").count()} rows')

In [ ]:
# Silver enrolments — cast, flag
silver_enrolments = (
    df_enrolments
    .withColumn('enrolment_date', to_date('enrolment_date'))
    .withColumn('credits',        col('credits').cast('int'))
    .withColumn('is_completed',   col('is_completed').cast('int'))
    .withColumn('is_withdrawn',   col('is_withdrawn').cast('int'))
    .withColumn('silver_timestamp', current_timestamp())
)
silver_enrolments.write.mode('overwrite').format('delta').saveAsTable('silver_enrolments')
print(f'Silver enrolments: {spark.table("silver_enrolments").count()} rows')

In [ ]:
# Silver assessments — cast score, derive grade band, retention risk
silver_assessments = (
    df_assessments
    .withColumn('submitted_date', to_date('submitted_date'))
    .withColumn('score',          col('score').cast('double'))
    .withColumn('is_pass',        col('is_pass').cast('int'))
    .withColumn('attempt_number', col('attempt_number').cast('int'))
    .withColumn('grade_band',
        when(col('score') >= 70, 'Distinction')
        .when(col('score') >= 60, 'Merit')
        .when(col('score') >= 50, 'Pass')
        .when(col('score') >= 40, 'Near Pass')
        .otherwise('Fail')
    )
    .withColumn('is_resit',
        when(col('attempt_number') > 1, 1).otherwise(0)
    )
    .withColumn('silver_timestamp', current_timestamp())
)
silver_assessments.write.mode('overwrite').format('delta').saveAsTable('silver_assessments')
print(f'Silver assessments: {spark.table("silver_assessments").count()} rows')

In [ ]:
print('Silver transformation complete.')
print(f'  Faculty     : {silver_faculty.count():>7,}')
print(f'  Students    : {silver_students.count():>7,}')
print(f'  Enrolments  : {silver_enrolments.count():>7,}')
print(f'  Assessments : {silver_assessments.count():>7,}')